## Feature Engineering

### Add features

text_length → number of characters

word_count → number of words

has_error_keyword

has_failure_keyword

has_urgent_keyword

has_access_keyword

has_payment_keyword

has_refund_keyword

> These are all based on the issue_description.

In [13]:
# Imports
import pandas as pd
from pathlib import Path

In [14]:
# Load raw dataset
raw_path = Path("../data/raw/aa_dataset-tickets-multi-lang-5-2-50-version.csv")

df = pd.read_csv(raw_path)

print("Raw dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Raw dataset shape: (28587, 16)

Columns:
['subject', 'body', 'answer', 'type', 'queue', 'priority', 'language', 'version', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8']


In [15]:
# Preview the data
df.head(5)

,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51,Security,Outage,Disruption,Data Breach,NaN,NaN,NaN,NaN
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51,Account,Disruption,Outage,IT,Tech Support,NaN,NaN,NaN
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51,Product,Feature,Tech Support,NaN,NaN,NaN,NaN,NaN
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51,Billing,Payment,Account,Documentation,Feedback,NaN,NaN,NaN
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51,Product,Feature,Feedback,Tech Support,NaN,NaN,NaN,NaN


In [16]:
# Check missing values
if "language" in df.columns:
    print("Language distribution:\n")
    print(df["language"].value_counts(dropna=False))
else:
    print("No 'language' column found.")

Language distribution:

language
en    16338
de    12249
Name: count, dtype: int64


In [17]:
# Check language distribution
if "language" in df.columns:
    print("Language distribution:\n")
    print(df["language"].value_counts(dropna=False))
else:
    print("No 'language' column found.")

Language distribution:

language
en    16338
de    12249
Name: count, dtype: int64


In [18]:
# Keep only English rows
df_en = df[df["language"] == "en"].copy()

print("English-only dataset shape:", df_en.shape)

English-only dataset shape: (16338, 16)


In [19]:
# Inspect useful columns
useful_cols = ["subject", "body", "priority", "type", "queue", "language"]
existing_cols = [col for col in useful_cols if col in df_en.columns]

df_en[existing_cols].head(10)

,subject,body,priority,type,queue,language
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...",high,Incident,Technical Support,en
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",medium,Request,Returns and Exchanges,en
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",low,Request,Billing and Payments,en
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",medium,Problem,Sales and Pre-Sales,en
5,Feature Query,"Dear Customer Support,\n\nI hope this message ...",high,Request,Technical Support,en
6,System Interruptions,"Dear Customer Support Team,\n\nI am submitting...",high,Incident,Service Outages and Maintenance,en
7,Connectivity Problems with Printer on MacBook Pro,"Dear Support Team,\n\nI am reporting a recurri...",medium,Incident,Technical Support,en
10,VPN Access Issue,"Customer Support,\n\nWe are encountering a dis...",medium,Incident,Product Support,en
12,Immediate Help Needed: Technical Problem with ...,"Dear Customer Support Team,\n\nI am submitting...",medium,Problem,IT Support,en
13,Inquiry for Detailed Information on Agency Off...,"Dear Customer Support Team,\n\nI hope this mes...",high,Request,Product Support,en


In [20]:
# Combine subject and body into issue_description
df_en["subject"] = df_en["subject"].fillna("").astype(str)
df_en["body"] = df_en["body"].fillna("").astype(str)

df_en["issue_description"] = (
    df_en["subject"].str.strip() + " " + df_en["body"].str.strip()
).str.strip()

df_en[["subject", "body", "issue_description", "priority"]].head(5)

,subject,body,issue_description,priority
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Account Disruption Dear Customer Support Team,...",high
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Query About Smart Home System Integration Feat...,medium
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",Inquiry Regarding Invoice Details Dear Custome...,low
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Question About Marketing Agency Software Compa...,medium
5,Feature Query,"Dear Customer Support,\n\nI hope this message ...","Feature Query Dear Customer Support,\n\nI hope...",high


In [21]:
# Keep required columns
ml_df = df_en[["issue_description", "type", "queue", "priority"]].copy()

# -------------------------------
# Basic cleaning
# -------------------------------
ml_df = ml_df.dropna(subset=["issue_description", "priority"]).copy()

ml_df["issue_description"] = ml_df["issue_description"].astype(str).str.strip()
ml_df["type"] = ml_df["type"].fillna("UNKNOWN").astype(str).str.strip().str.upper()
ml_df["queue"] = ml_df["queue"].fillna("UNKNOWN").astype(str).str.strip().str.upper()
ml_df["priority"] = ml_df["priority"].astype(str).str.strip().str.upper()

ml_df = ml_df[ml_df["issue_description"] != ""].copy()
ml_df = ml_df[ml_df["issue_description"].str.len() > 20].copy()

# -------------------------------
# Feature engineering
# -------------------------------

text_series = ml_df["issue_description"].str.lower()

# Numeric features
ml_df["text_length"] = ml_df["issue_description"].str.len()
ml_df["word_count"] = ml_df["issue_description"].str.split().str.len()

# Keyword groups
error_keywords = ["error", "bug", "issue", "problem", "crash"]
failure_keywords = ["failed", "failure", "not working", "broken", "unable"]
urgent_keywords = ["urgent", "immediately", "asap", "critical", "emergency"]
access_keywords = ["login", "account", "password", "access", "authentication"]
payment_keywords = ["payment", "charged", "billing", "transaction", "invoice"]

# Binary keyword features
ml_df["has_error_keyword"] = text_series.apply(
    lambda x: int(any(k in x for k in error_keywords))
)

ml_df["has_failure_keyword"] = text_series.apply(
    lambda x: int(any(k in x for k in failure_keywords))
)

ml_df["has_urgent_keyword"] = text_series.apply(
    lambda x: int(any(k in x for k in urgent_keywords))
)

ml_df["has_access_keyword"] = text_series.apply(
    lambda x: int(any(k in x for k in access_keywords))
)

ml_df["has_payment_keyword"] = text_series.apply(
    lambda x: int(any(k in x for k in payment_keywords))
)

# -------------------------------
# Final Stage 4 dataset
# -------------------------------
ml_df_final = ml_df[
    [
        "issue_description",
        "type",
        "queue",
        "text_length",
        "word_count",
        "has_error_keyword",
        "has_failure_keyword",
        "has_urgent_keyword",
        "has_access_keyword",
        "has_payment_keyword",
        "priority",
    ]
].copy()

print("Stage 4 dataset shape:", ml_df_final.shape)
print(ml_df_final.head())

# -------------------------------
# Save dataset
# -------------------------------
processed_path = Path("../data/processed/ml_priority_dataset_stage4.csv")
processed_path.parent.mkdir(parents=True, exist_ok=True)

ml_df_final.to_csv(processed_path, index=False)
print(f"Saved Stage 4 dataset to: {processed_path}")

Stage 4 dataset shape: (16337, 11)
                                   issue_description      type  \
1  Account Disruption Dear Customer Support Team,...  INCIDENT   
2  Query About Smart Home System Integration Feat...   REQUEST   
3  Inquiry Regarding Invoice Details Dear Custome...   REQUEST   
4  Question About Marketing Agency Software Compa...   PROBLEM   
5  Feature Query Dear Customer Support,\n\nI hope...   REQUEST   

                   queue  text_length  word_count  has_error_keyword  \
1      TECHNICAL SUPPORT          563          84                  1   
2  RETURNS AND EXCHANGES          585          83                  0   
3   BILLING AND PAYMENTS          639          95                  0   
4    SALES AND PRE-SALES          732         103                  0   
5      TECHNICAL SUPPORT          660          99                  0   

   has_failure_keyword  has_urgent_keyword  has_access_keyword  \
1                    0                   0                   1   
2  

In [23]:
#Sanity checks
print(ml_df_final["priority"].value_counts())
print(ml_df_final[["text_length", "word_count"]].describe())

for col in [
    "has_error_keyword",
    "has_failure_keyword",
    "has_urgent_keyword",
    "has_access_keyword",
    "has_payment_keyword",
]:
    print(f"\n{col}")
    print(ml_df_final[col].value_counts())

priority
MEDIUM    6617
HIGH      6346
LOW       3374
Name: count, dtype: int64
        text_length    word_count
count  16337.000000  16337.000000
mean     405.400135     58.646447
std      179.656656     26.865849
min       24.000000      2.000000
25%      255.000000     36.000000
50%      416.000000     60.000000
75%      565.000000     82.000000
max     1189.000000    172.000000

has_error_keyword
has_error_keyword
1    8472
0    7865
Name: count, dtype: int64

has_failure_keyword
has_failure_keyword
0    15418
1      919
Name: count, dtype: int64

has_urgent_keyword
has_urgent_keyword
0    14861
1     1476
Name: count, dtype: int64

has_access_keyword
has_access_keyword
0    13334
1     3003
Name: count, dtype: int64

has_payment_keyword
has_payment_keyword
0    15386
1      951
Name: count, dtype: int64
